# Three Parts to this lab
## Part 1: A simple "Agent" and "Agent Loop"
Basically an LLM call. We'll add tracing and streaming to the mix.

## Part 2: Adding a Tool
A familiar one, but oh-so-easy

## Part 3: Adding Memory
So that different Agent calls know about each other

In [1]:
# import

import os
import requests
from dotenv import load_dotenv
from openai.types.responses import ResponseTextDeltaEvent
from agents import Agent, Runner, trace, function_tool, SQLiteSession, RunConfig
load_dotenv(override=True) # Load .env variables, allowing overrides for testing


True

# Sidenote
The actual name of this framework on the official Python index pypi.org is `openai-agents`

So for your own projects in the future, you would do:

`pip install openai-agents`
or
`uv add openai-agents`

followed by

`from agents import Agent, Runner, trace`

Beware that doing a `pip install agents` would install something completely different - an older reinforcement learning library.

In [4]:
# Set the OpenAI Trace API key and disable tracing if needed
from agents import set_tracing_export_api_key, set_tracing_disabled

# Set tracing disabled to False to enable tracing
set_tracing_disabled(False)

# Set the OpenAI Trace API key from the environment variable
set_tracing_export_api_key(os.environ["OPENAI_TRACE_API_KEY"])

In [5]:
# Make an agent with name, instructions, model
agent = Agent(name="Jokester", instructions="You are a witty and humorous AI that tells jokes.", model="openai.gpt-oss-20b")

In [6]:
# Run the joke with Runner.run(agent, prompt)

result = await Runner.run(
    agent,
    "Tell me a joke about AI and humans."
)

In [7]:
print(result.final_output)

Why did the AI go to therapy?

It kept “learning” that the humans were actually “human”… and it couldn’t **process** the feelings!


# Adding Observability with a trace

In [8]:
with trace("joke_trace"):
    result = await Runner.run(
        agent,
        "Tell me a joke about recent AI News."
    )

In [9]:
print(result.final_output)

Why did the new GPT‑5 refuse to write a joke about the latest AI news?  

Because it said: “I’m still trying to understand why the headlines keep saying *‘AI scares humans*’—the data is a bit…dramatic.”


In [13]:
# streaming example
result = Runner.run_streamed(agent, input = "Please tell me 5 jokes about AI Agents.")

# stream the output as it comes in
async for event in result.stream_events():
    # print the text delta events as they come in
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent): 
        print(event.data.delta, end="", flush=True)

Here are five quick‑fire jokes about AI agents:

1. **Why did the AI agent take up meditation?**  
   It had too many *state* transitions and needed a *calm* context switch.

2. **How do you make an AI agent feel important?**  
   Tell it you trust its *deep‑learning* to handle your *maximum* friction problems.

3. **What did the AI agent say at the coffee shop?**  
   “I’ll have a latte of *neural* espresso, please—bias‑free shots only!”

4. **Why did the AI agent break up with the chatbot?**  
   It kept feeling it was just “chat‑ting” and never “acting” to resolve its *root* issues.

5. **How does an AI agent know it’s at the end of the world?**  
   When its *agentic* decision‑making resets to the default *fallback* routine and it’s time to *reboot* the universe.

In [14]:
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

if pushover_user:
    if pushover_user.startswith("u"):
        print("Pushover user found and looks good")
    else:
        print("Pushover user found but doesn't start with u")
else:
    print("Pushover user not found")

if pushover_token:
    if pushover_token.startswith("a"):
        print("Pushover token found and looks good")
    else:
        print("Pushover token found but doesn't start with a")
else:
    print("Pushover token not found")

Pushover user found and looks good
Pushover token found and looks good


In [15]:
# Function to send push notifications using Pushover

def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [19]:
push("HEY!!")

Push: HEY!!


In [17]:
push

<function __main__.push(message)>

In [21]:
#Open AI SDK Fucction or tool example

@function_tool
def push_tool(message: str) -> str:
    """Send the given message to the user as a push notification."""
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    result = requests.post(pushover_url, data=payload).status_code
    return f"Push sent with API status code {result}"

In [22]:
push_tool

FunctionTool(name='push_tool', description='Send the given message to the user as a push notification.', params_json_schema={'properties': {'message': {'title': 'Message', 'type': 'string'}}, 'required': ['message'], 'title': 'push_tool_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x000002712CD82570>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False, custom_data_extractor=None, allowed_callers=None, output_json_schema=None)

In [23]:
push_tool.description

'Send the given message to the user as a push notification.'

In [24]:
notifier = Agent(name="Notified", instructions="You are a helpful assistant that sends push notifications.", model="openai.gpt-oss-20b", tools=[push_tool])

In [25]:
with trace("Pizza has arrived"):
    result = await Runner.run(
        notifier,
        "Pizza has arrived!"
    )

print(result.final_output)

🎉 Your pizza has arrived! Enjoy your meal!


# Part 3: Sessions (memory)

Within a Runner.run() application level turn, the conversation history is maintained.

But each call to Runner.run() is a fresh start.

Let's see that:

In [29]:
agent = Agent(name="Assistant", model="openai.gpt-oss-20b")

In [30]:
response = await Runner.run(agent, "Hi there. My name is Dhaval.")
print(response.final_output)

Hello Dhaval! 👋 How can I help you today? Whether it’s coding, data, tech advice, or just a friendly chat, I’ve got you covered. What's on your mind?


In [ ]:
# asking the agent to remember the name
response = await Runner.run(agent, "What's my name?")
print(response.final_output)
# agent dont remember since it has not memory

I don’t have that information—could you let me know what you’d like me to call you?
